# Task 0: The Library of Babel
## Building a Dataset with Three Distinct Classes

**Goal**: Create a dataset where authorship (human vs AI) is the primary variable, NOT topic.

### Classes:
1. **Class 1**: Human-written text (from Project Gutenberg authors)
2. **Class 2**: AI-generated text on same topics (Gemini, neutral style)
3. **Class 3**: AI-generated text mimicking the author's style

In [1]:
# Install required libraries
!pip install requests pandas numpy google-generativeai nltk spacy textstat scikit-learn -q
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 44.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create a project folder
PROJECT_PATH = '/content/drive/MyDrive/Precog_NLP_Task'
if not os.path.exists(PROJECT_PATH):
    os.makedirs(PROJECT_PATH)
    print(f"Created directory: {PROJECT_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import requests
import pandas as pd
import numpy as np
import re
import time
import google.generativeai as genai
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## Step 1: Configure Gemini API

In [4]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemma-3-27b-it')

print("✓ Gemini API configured successfully")

✓ Gemini API configured successfully


## Step 2: Fetch and Clean Human Text (Class 1)

In [5]:
def get_clean_text(url):
    """
    Fetches text from Project Gutenberg and removes header/footer boilerplate.
    """
    response = requests.get(url)
    response.raise_for_status()
    text = response.text

    text = re.sub(r'’', '\'', text, flags=re.DOTALL)
    text = re.sub(r'_', '', text, flags=re.DOTALL)
    text = re.sub(r'—', ' ', text, flags=re.DOTALL)
    text = re.sub(r'“', '\"', text, flags=re.DOTALL)
    text = re.sub(r'\[_Copyright.*?\]', '', text, flags=re.DOTALL)
    text = re.sub(r'\[Illustration.*?\]', '', text, flags=re.DOTALL)
    text = re.sub(r',', ',', text, flags=re.DOTALL)
    text = re.sub(r'!', '!', text, flags=re.DOTALL)

    # Remove Project Gutenberg header
    start_markers = [
        "*** START OF THIS PROJECT GUTENBERG",
        "*** START OF THE PROJECT GUTENBERG",
        "***START OF THE PROJECT GUTENBERG"
    ]
    for marker in start_markers:
        if marker in text:
            text = text.split(marker)[1]
            break

    # Remove Project Gutenberg footer
    end_markers = [
        "*** END OF THIS PROJECT GUTENBERG",
        "*** END OF THE PROJECT GUTENBERG",
        "***END OF THE PROJECT GUTENBERG",
        "End of the Project Gutenberg"
    ]
    for marker in end_markers:
        if marker in text:
            text = text.split(marker)[0]
            break

    lines = text.split('\n')
    cleaned_lines = [l for l in lines if not any(word in l.upper() for word in ["GUTENBERG", "TRANSCRIBER"])]
    text = '\n'.join(cleaned_lines)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text.strip()


def create_chunks(text, author_name, min_words=100, max_words=200, overlap=20):
    """
    Splits text into chunks of 100-200 words with slight overlap.

    Args:
        text: The source text to chunk
        author_name: Name of the author (for labeling)
        min_words: Minimum words per chunk
        max_words: Maximum words per chunk
        overlap: Number of words to overlap between chunks
    """
    # Split into sentences (basic approach)
    sentences = re.split(r'(?<=[.!?])\s+', text)

    chunks = []
    current_chunk = []
    current_word_count = 0

    for sentence in sentences:
        words = sentence.split()
        word_count = len(words)

        # If adding this sentence keeps us under max_words, add it
        if current_word_count + word_count <= max_words:
            current_chunk.append(sentence)
            current_word_count += word_count
        else:
            # Save current chunk if it meets minimum
            if current_word_count >= min_words:
                chunks.append(' '.join(current_chunk))

            # Start new chunk with overlap
            if overlap > 0 and len(current_chunk) > 0:
                # Keep last few sentences for overlap
                overlap_text = ' '.join(current_chunk[-2:]) if len(current_chunk) >= 2 else current_chunk[-1]
                overlap_words = overlap_text.split()
                if len(overlap_words) > overlap:
                    overlap_text = ' '.join(overlap_words[-overlap:])
                current_chunk = [overlap_text, sentence]
                current_word_count = len(overlap_text.split()) + word_count
            else:
                current_chunk = [sentence]
                current_word_count = word_count

    # Don't forget the last chunk
    if current_word_count >= min_words:
        chunks.append(' '.join(current_chunk))

    # Create DataFrame
    df = pd.DataFrame({
        'text': chunks,
        'author': author_name,
        'class': 'human',
        'word_count': [len(chunk.split()) for chunk in chunks]
    })

    return df


def get_bulk_class1(url, start_anchor, char_limit=200000):
    """
    Fetches a large chunk of text starting from a specific anchor point.
    """
    text = get_clean_text(url)
    start_idx = text.find(start_anchor)

    if start_idx == -1:
        print(f"Warning: Anchor '{start_anchor}' not found. Using text from beginning.")
        start_idx = 0

    return text[start_idx : start_idx + char_limit]

In [6]:
# Fetch texts from two authors
print("Fetching Jane Austen - Pride and Prejudice...")
austen_bulk = get_bulk_class1(
    "https://www.gutenberg.org/cache/epub/1342/pg1342.txt",
    "It is a truth universally"
)

print("Fetching Charles Dickens - Great Expectations...")
dickens_bulk = get_bulk_class1(
    "https://www.gutenberg.org/cache/epub/1400/pg1400.txt",
    "My father's family"
)

# Create chunks
print("\nCreating text chunks...")
df_austen = create_chunks(austen_bulk, "Austen")
df_dickens = create_chunks(dickens_bulk, "Dickens")

# Combine into Class 1 dataset
class1_df = pd.concat([df_austen, df_dickens], ignore_index=True)

print(f"\n✓ Class 1 (Human) Dataset Created:")
print(f"  - Total samples: {len(class1_df)}")
print(f"  - Austen samples: {len(df_austen)}")
print(f"  - Dickens samples: {len(df_dickens)}")
print(f"  - Avg words per sample: {class1_df['word_count'].mean():.1f}")
print(f"  - Word count range: {class1_df['word_count'].min()}-{class1_df['word_count'].max()}")

Fetching Jane Austen - Pride and Prejudice...
Fetching Charles Dickens - Great Expectations...

Creating text chunks...

✓ Class 1 (Human) Dataset Created:
  - Total samples: 445
  - Austen samples: 218
  - Dickens samples: 227
  - Avg words per sample: 182.9
  - Word count range: 105-200


In [7]:
# Preview some samples
print("\n=== Sample from Austen ===")
print(class1_df[class1_df['author'] == 'Austen'].iloc[10]['text'][:300] + "...")

print("\n=== Sample from Dickens ===")
print(class1_df[class1_df['author'] == 'Dickens'].iloc[0]['text'][:300] + "...")


=== Sample from Austen ===
five daughters, could ask on the subject, was sufficient to draw from her husband any satisfactory description of Mr. Bingley. They attacked him in various ways, with barefaced questions, ingenious suppositions, and distant surmises; but he eluded the skill of them all; and they were at last obliged...

=== Sample from Dickens ===
My father's family name being Pirrip, and my Christian name Philip, my infant tongue could make of both names nothing longer or more explicit than Pip. So, I called myself Pip, and came to be called Pip. I give Pirrip as my father's family name, on the authority of his tombstone and my sister, Mrs. ...


## Step 3: Extract Topics from Human Text

In [8]:
def extract_topics_with_gemini(text_sample, num_topics, start, end):
    """
    Uses Gemini to extract core topics from the text.
    """
    prompt = f"""
    <start_of_turn>user
    You are an expert Literary Scholar specializing in thematic decomposition.

    TASK:
    Analyze the provided literary excerpt and extract exactly {num_topics} core substantive themes. It is imperative that these themes must be central to the text.

    CONSTRAINTS:
    - Focus exclusively on abstract, higher-order themes (e.g., "social hierarchy," "unrequited affection," "industrial alienation").
    - DO NOT list plot points, character names, or specific events.
    - Each theme must be a concise 2-3 word label.
    - Output ONLY a numbered list. No introductory or concluding remarks.

    TEXT EXCERPT:
    ---
    {text_sample[start:end]}
    ---

    OUTPUT FORMAT:
    1. [Theme Name]
    2. [Theme Name]
    ...
    <end_of_turn>
    <start_of_turn>model
    """

    try:
        response = model.generate_content(prompt)
        topics_text = response.text

        # Parse the numbered list
        topics = []
        for line in topics_text.split('\n'):
            # Match lines like "1. Topic" or "1) Topic"
            match = re.match(r'^\d+[.)\s]+(.*?)$', line.strip())
            if match:
                topic = match.group(1).strip()
                if topic:
                    topics.append(topic)

        return topics[:num_topics]

    except Exception as e:
        print(f"Error extracting topics: {e}")
        return []


# Extract topics from both authors
print("Extracting topics using Gemini...\n")

# Sample text from each author for topic extraction
austen_sample = ' '.join(class1_df[class1_df['author'] == 'Austen']['text'].head(250))
dickens_sample = ' '.join(class1_df[class1_df['author'] == 'Dickens']['text'].head(250))

topics_austen, topics_dickens = [], []

for i in range(2):
  topicsAusten = extract_topics_with_gemini(austen_sample, num_topics=5, start=i*5000, end=i*10000+4000)
  time.sleep(2)  # Rate limiting
  topicsDickens = extract_topics_with_gemini(dickens_sample, num_topics=5, start=i*5000, end=i*10000+4000)
  time.sleep(2)
  topics_austen.extend(topicsAusten), topics_dickens.extend(topicsDickens)

all_topics = topics_austen + topics_dickens

print(f"\n✓ Extracted {len(all_topics)} topics:")
for i, topic in enumerate(all_topics, 1):
    print(f"  {i}. {topic}")

# Save topics for later use
topics_list = all_topics

Extracting topics using Gemini...


✓ Extracted 20 topics:
  1. Social Class
  2. Marriage Prospects
  3. Parental Duty
  4. Gender Roles
  5. Superficiality
  6. Social Expectations
  7. Class Consciousness
  8. Parental Influence
  9. Marital Prospects
  10. Superficiality
  11. Social Isolation
  12. Childhood Trauma
  13. Class Disparity
  14. Mortality Awareness
  15. Identity Formation
  16. Power Dynamics
  17. Social Control
  18. Existential Threat
  19. Psychological Manipulation
  20. Childhood Vulnerability


## Step 4: Generate AI Text - Class 2 (Neutral Style)

In [9]:
from prompt import *

In [10]:
# Generate Class 2: AI text in neutral style
print("\n" + "="*60)
print("GENERATING CLASS 2: AI TEXT (NEUTRAL STYLE)")
print("="*60)

class2_df = generate_ai_dataset_optimized(
    model = model,
    topics=topics_list,
    num_samples_per_topic=25,  # 25 samples per topic
    style="neutral",
    batch_size=5
)

print(f"\n✓ Class 2 (AI Neutral) Dataset Created:")
print(f"  - Total samples: {len(class2_df)}")
print(f"  - Avg words per sample: {class2_df['word_count'].mean():.1f}")
print(f"  - Word count range: {class2_df['word_count'].min()}-{class2_df['word_count'].max()}")

check_dataset_diversity(class2_df)


GENERATING CLASS 2: AI TEXT (NEUTRAL STYLE)
VARIETY MODE ENABLED
 - Using varied batch generation to minimize duplicates
 - Explicit uniqueness instructions per batch

Generating 500 samples (25 per topic)...
Using batch generation: 5 paragraphs per API call
Total API calls needed: 100 (vs 500 without batching)
Time savings: ~10.0 minutes


TOPIC 1/20: Social Class
  Generating 25 VARIED samples...
  API calls needed: 5 (batch size: 5)
  API Call 1/5: Generating 5 variations... ✓ (5/25 total)
  API Call 2/5: Generating 5 variations... ✓ (10/25 total)
  API Call 3/5: Generating 5 variations...

ERROR:tornado.access:503 POST /v1beta/models/gemma-3-27b-it:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 811.01ms


 ✓ (15/25 total)
  API Call 4/5: Generating 5 variations... ✓ (20/25 total)
  API Call 5/5: Generating 5 variations... ✓ (25/25 total)

TOPIC 2/20: Marriage Prospects
  Generating 25 VARIED samples...
  API calls needed: 5 (batch size: 5)
  API Call 1/5: Generating 5 variations... ✓ (5/25 total)
  API Call 2/5: Generating 5 variations...

ERROR:tornado.access:503 POST /v1beta/models/gemma-3-27b-it:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 888.29ms
ERROR:tornado.access:503 POST /v1beta/models/gemma-3-27b-it:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 16871.99ms


 ✓ (10/25 total)
  API Call 3/5: Generating 5 variations... ✓ (15/25 total)
  API Call 4/5: Generating 5 variations... ✓ (20/25 total)
  API Call 5/5: Generating 5 variations... ✓ (25/25 total)

TOPIC 3/20: Parental Duty
  Generating 25 VARIED samples...
  API calls needed: 5 (batch size: 5)
  API Call 1/5: Generating 5 variations...

...

 ✓ (20/25 total)
  API Call 5/5: Generating 5 variations... ✓ (25/25 total)

TOPIC 17/20: Social Control
  Generating 25 VARIED samples...
  API calls needed: 5 (batch size: 5)
  API Call 1/5: Generating 5 variations... ✓ (5/25 total)
  API Call 2/5: Generating 5 variations... ✓ (10/25 total)
  API Call 3/5: Generating 5 variations... ✓ (15/25 total)
  API Call 4/5: Generating 5 variations... ✓ (20/25 total)
  API Call 5/5: Generating 5 variations... ✓ (25/25 total)

TOPIC 18/20: Existential Threat
  Generating 25 VARIED samples...
  API calls needed: 5 (batch size: 5)
  API Call 1/5: Generating 5 variations...

ERROR:tornado.access:503 POST /v1beta/models/gemma-3-27b-it:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1545.95ms
ERROR:tornado.access:503 POST /v1beta/models/gemma-3-27b-it:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 12757.14ms


 ✓ (5/25 total)
  API Call 2/5: Generating 5 variations... ✓ (10/25 total)
  API Call 3/5: Generating 5 variations... ✓ (15/25 total)
  API Call 4/5: Generating 5 variations... ✓ (20/25 total)
  API Call 5/5: Generating 5 variations... ✓ (25/25 total)

TOPIC 19/20: Psychological Manipulation
  Generating 25 VARIED samples...
  API calls needed: 5 (batch size: 5)
  API Call 1/5: Generating 5 variations... ✓ (5/25 total)
  API Call 2/5: Generating 5 variations... ✓ (10/25 total)
  API Call 3/5: Generating 5 variations... ✓ (15/25 total)
  API Call 4/5: Generating 5 variations... ✓ (20/25 total)
  API Call 5/5: Generating 5 variations... ✓ (25/25 total)

TOPIC 20/20: Childhood Vulnerability
  Generating 25 VARIED samples...
  API calls needed: 5 (batch size: 5)
  API Call 1/5: Generating 5 variations... ✓ (5/25 total)
  API Call 2/5: Generating 5 variations... ✓ (10/25 total)
  API Call 3/5: Generating 5 variations...

ERROR:tornado.access:503 POST /v1beta/models/gemma-3-27b-it:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3469.40ms


 ✓ (15/25 total)
  API Call 4/5: Generating 5 variations... ✓ (20/25 total)
  API Call 5/5: Generating 5 variations... ✓ (25/25 total)

✓ Generation complete!
  - Successfully generated: 500 samples
  - Failed generations: 0
  - Exact duplicates found: 2
  Consider using variety mode or reducing batch size

✓ Class 2 (AI Neutral) Dataset Created:
  - Total samples: 500
  - Avg words per sample: 110.7
  - Word count range: 90-138

DIVERSITY ANALYSIS

1. EXACT DUPLICATES: 2
   Found 2 exact duplicates
   Sample duplicate:
   'Is it possible to build a life on shimmering surfaces, devoid of genuine connection? Superficiality, at its core, isn’t simply about appearances; it’s...'

2. NEAR-DUPLICATES (same first 50 words): 25

3. TOPIC-LEVEL VARIETY:
   - 'Social Class...': 0 duplicates out of 25 samples
   - 'Marriage Prospects...': 0 duplicates out of 25 samples
   - 'Parental Duty...': 0 duplicates out of 25 samples

4. VOCABULARY DIVERSITY:
   - Type-Token Ratio (first 100 samples): 0.2

In [11]:
# Preview Class 2 samples
print("\n=== Sample AI Text (Neutral Style) ===")
print(class2_df.iloc[490]['text'])


=== Sample AI Text (Neutral Style) ===
Isn't it strange how the world feels infinitely larger when you’re small? Childhood vulnerability isn’t simply about physical helplessness, though that’s certainly a component. It’s the profound lack of agency, the complete reliance on others for interpretation of reality. A scraped knee isn’t just pain; it’s a catastrophic injury demanding the immediate intervention of a trusted adult, whose reaction dictates the severity of the experience. This dependence extends beyond the physical, encompassing emotional and psychological needs. A child’s sense of self is fragile, easily molded by praise or criticism, and lacking the internal framework to critically assess external input. Consequently, vulnerability in childhood is a state of radical openness, a susceptibility to influence that shapes the very foundations of personality.


## Step 5: Generate AI Text - Class 3 (Mimicked Style)

In [12]:
# Generate Class 3: AI text mimicking Austen's style
print("\n" + "="*60)
print("GENERATING CLASS 3A: AI TEXT (MIMICKING AUSTEN)")
print("="*60)

class3a_df = generate_ai_dataset_optimized(
    model,
    topics=topics_austen,
    num_samples_per_topic=25,  # 50 samples per topic
    style="mimicked",
    author_name="Austen",
    batch_size=5
)

print(f"\n✓ Class 3A (AI Mimicking Austen) Dataset Created:")
print(f"  - Total samples: {len(class3a_df)}")
print(f"  - Avg words per sample: {class3a_df['word_count'].mean():.1f}")

check_dataset_diversity(class3a_df)


GENERATING CLASS 3A: AI TEXT (MIMICKING AUSTEN)
VARIETY MODE ENABLED
 - Using varied batch generation to minimize duplicates
 - Explicit uniqueness instructions per batch

Generating 250 samples (25 per topic)...
Using batch generation: 5 paragraphs per API call
Total API calls needed: 50 (vs 250 without batching)
Time savings: ~5.0 minutes


TOPIC 1/10: Social Class
  Generating 25 VARIED samples...
  API calls needed: 5 (batch size: 5)
  API Call 1/5: Generating 5 variations... ✓ (5/25 total)
  API Call 2/5: Generating 5 variations... ✓ (10/25 total)
  API Call 3/5: Generating 5 variations... ✓ (15/25 total)
  API Call 4/5: Generating 5 variations... ✓ (20/25 total)
  API Call 5/5: Generating 5 variations... ✓ (25/25 total)

TOPIC 2/10: Marriage Prospects
  Generating 25 VARIED samples...
  API calls needed: 5 (batch size: 5)
  API Call 1/5: Generating 5 variations... ✓ (5/25 total)
  API Call 2/5: Generating 5 variations... ✓ (10/25 total)
  API Call 3/5: Generating 5 variations...

ERROR:tornado.access:503 POST /v1beta/models/gemma-3-27b-it:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 24333.47ms
ERROR:tornado.access:503 POST /v1beta/models/gemma-3-27b-it:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1448.68ms


 ✓ (15/25 total)
  API Call 4/5: Generating 5 variations... ✓ (20/25 total)
  API Call 5/5: Generating 5 variations... ✓ (25/25 total)

TOPIC 3/10: Parental Duty
  Generating 25 VARIED samples...
  API calls needed: 5 (batch size: 5)
  API Call 1/5: Generating 5 variations... ✓ (5/25 total)
  API Call 2/5: Generating 5 variations... ✓ (10/25 total)
  API Call 3/5: Generating 5 variations... ✓ (15/25 total)
  API Call 4/5: Generating 5 variations... ✓ (20/25 total)
  API Call 5/5: Generating 5 variations... ✓ (25/25 total)

TOPIC 4/10: Gender Roles
  Generating 25 VARIED samples...
  API calls needed: 5 (batch size: 5)
  API Call 1/5: Generating 5 variations... ✓ (5/25 total)
  API Call 2/5: Generating 5 variations...

ERROR:tornado.access:503 POST /v1beta/models/gemma-3-27b-it:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 15351.20ms
ERROR:tornado.access:503 POST /v1beta/models/gemma-3-27b-it:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 836.97ms


 ✓ (10/25 total)
  API Call 3/5: Generating 5 variations... ✓ (15/25 total)
  API Call 4/5: Generating 5 variations... ✓ (20/25 total)
  API Call 5/5: Generating 5 variations... ✓ (25/25 total)

TOPIC 5/10: Superficiality
  Generating 25 VARIED samples...
  API calls needed: 5 (batch size: 5)
  API Call 1/5: Generating 5 variations... ✓ (5/25 total)
  API Call 2/5: Generating 5 variations... ✓ (10/25 total)
  API Call 3/5: Generating 5 variations... ✓ (15/25 total)
  API Call 4/5: Generating 5 variations... ✓ (20/25 total)
  API Call 5/5: Generating 5 variations... ✓ (25/25 total)

TOPIC 6/10: Social Expectations
  Generating 25 VARIED samples...
  API calls needed: 5 (batch size: 5)
  API Call 1/5: Generating 5 variations... ✓ (5/25 total)
  API Call 2/5: Generating 5 variations... ✓ (10/25 total)
  API Call 3/5: Generating 5 variations... ✓ (15/25 total)
  API Call 4/5: Generating 5 variations... ✓ (20/25 total)
  API Call 5/5: Generating 5 variations... ✓ (25/25 total)

TOPIC 7/10: 

...

 ✓ (5/25 total)
  API Call 2/5: Generating 5 variations... ✓ (10/25 total)
  API Call 3/5: Generating 5 variations... ✓ (15/25 total)
  API Call 4/5: Generating 5 variations... ✓ (20/25 total)
  API Call 5/5: Generating 5 variations...

ERROR:tornado.access:503 POST /v1beta/models/gemma-3-27b-it:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1696.75ms


 ✓ (25/25 total)

TOPIC 10/10: Superficiality
  Generating 25 VARIED samples...
  API calls needed: 5 (batch size: 5)
  API Call 1/5: Generating 5 variations... ✓ (5/25 total)
  API Call 2/5: Generating 5 variations... ✓ (10/25 total)
  API Call 3/5: Generating 5 variations...

ERROR:tornado.access:503 POST /v1beta/models/gemma-3-27b-it:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 811.89ms
ERROR:tornado.access:503 POST /v1beta/models/gemma-3-27b-it:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 863.57ms
ERROR:tornado.access:503 POST /v1beta/models/gemma-3-27b-it:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 785.89ms


 ✓ (15/25 total)
  API Call 4/5: Generating 5 variations... ✓ (20/25 total)
  API Call 5/5: Generating 5 variations... ✓ (25/25 total)

✓ Generation complete!
  - Successfully generated: 250 samples
  - Failed generations: 0
  - Exact duplicates found: 5
  Consider using variety mode or reducing batch size

✓ Class 3A (AI Mimicking Austen) Dataset Created:
  - Total samples: 250
  - Avg words per sample: 129.7

DIVERSITY ANALYSIS

1. EXACT DUPLICATES: 5
   Found 5 exact duplicates
   Sample duplicate:
   'It is a truth universally acknowledged, that a single fortune, though never so modest, must be the object of every young woman of good family; yet, it...'

2. NEAR-DUPLICATES (same first 50 words): 30

3. TOPIC-LEVEL VARIETY:
   - 'Social Class...': 0 duplicates out of 25 samples
   - 'Marriage Prospects...': 0 duplicates out of 25 samples
   - 'Parental Duty...': 0 duplicates out of 25 samples

4. VOCABULARY DIVERSITY:
   - Type-Token Ratio (first 100 samples): 0.250

5. SAMPLE VARIE

In [13]:
# Generate Class 3: AI text mimicking Dickens's style
print("\n" + "="*60)
print("GENERATING CLASS 3B: AI TEXT (MIMICKING DICKENS)")
print("="*60)

class3b_df = generate_ai_dataset_optimized(
    model,
    topics=topics_dickens,
    num_samples_per_topic=25,  # 50 samples per topic
    style="mimicked",
    author_name="Dickens",
    batch_size=5
)

print(f"\n✓ Class 3B (AI Mimicking Dickens) Dataset Created:")
print(f"  - Total samples: {len(class3b_df)}")
print(f"  - Avg words per sample: {class3b_df['word_count'].mean():.1f}")

check_dataset_diversity(class3b_df)


GENERATING CLASS 3B: AI TEXT (MIMICKING DICKENS)
VARIETY MODE ENABLED
 - Using varied batch generation to minimize duplicates
 - Explicit uniqueness instructions per batch

Generating 250 samples (25 per topic)...
Using batch generation: 5 paragraphs per API call
Total API calls needed: 50 (vs 250 without batching)
Time savings: ~5.0 minutes


TOPIC 1/10: Social Isolation
  Generating 25 VARIED samples...
  API calls needed: 5 (batch size: 5)
  API Call 1/5: Generating 5 variations... ✓ (5/25 total)
  API Call 2/5: Generating 5 variations...

ERROR:tornado.access:503 POST /v1beta/models/gemma-3-27b-it:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 8783.78ms
ERROR:tornado.access:503 POST /v1beta/models/gemma-3-27b-it:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1494.67ms


 ✓ (10/25 total)
  API Call 3/5: Generating 5 variations... ✓ (15/25 total)
  API Call 4/5: Generating 5 variations... ✓ (20/25 total)
  API Call 5/5: Generating 5 variations... ✓ (25/25 total)

TOPIC 2/10: Childhood Trauma
  Generating 25 VARIED samples...
  API calls needed: 5 (batch size: 5)
  API Call 1/5: Generating 5 variations... ✓ (5/25 total)
  API Call 2/5: Generating 5 variations... ✓ (10/25 total)
  API Call 3/5: Generating 5 variations... ✓ (15/25 total)
  API Call 4/5: Generating 5 variations... ✓ (20/25 total)
  API Call 5/5: Generating 5 variations... ✓ (25/25 total)

TOPIC 3/10: Class Disparity
  Generating 25 VARIED samples...
  API calls needed: 5 (batch size: 5)
  API Call 1/5: Generating 5 variations...

ERROR:tornado.access:503 POST /v1beta/models/gemma-3-27b-it:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 786.02ms


 ✓ (5/25 total)
  API Call 2/5: Generating 5 variations... ✓ (10/25 total)
  API Call 3/5: Generating 5 variations... ✓ (15/25 total)
  API Call 4/5: Generating 5 variations... ✓ (20/25 total)
  API Call 5/5: Generating 5 variations... ✓ (25/25 total)

TOPIC 4/10: Mortality Awareness
  Generating 25 VARIED samples...
  API calls needed: 5 (batch size: 5)
  API Call 1/5: Generating 5 variations... ✓ (5/25 total)
  API Call 2/5: Generating 5 variations... ✓ (10/25 total)
  API Call 3/5: Generating 5 variations... ✓ (15/25 total)
  API Call 4/5: Generating 5 variations... ✓ (20/25 total)
  API Call 5/5: Generating 5 variations... ✓ (25/25 total)

TOPIC 5/10: Identity Formation
  Generating 25 VARIED samples...
  API calls needed: 5 (batch size: 5)
  API Call 1/5: Generating 5 variations... ✓ (5/25 total)
  API Call 2/5: Generating 5 variations... ✓ (10/25 total)
  API Call 3/5: Generating 5 variations... ✓ (15/25 total)
  API Call 4/5: Generating 5 variations... ✓ (20/25 total)
  API Call

ERROR:tornado.access:503 POST /v1beta/models/gemma-3-27b-it:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 861.70ms
ERROR:tornado.access:503 POST /v1beta/models/gemma-3-27b-it:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 889.31ms


 ✓ (15/25 total)
  API Call 4/5: Generating 5 variations... ✓ (20/25 total)
  API Call 5/5: Generating 5 variations... ✓ (25/25 total)

TOPIC 8/10: Existential Threat
  Generating 25 VARIED samples...
  API calls needed: 5 (batch size: 5)
  API Call 1/5: Generating 5 variations... ✓ (5/25 total)
  API Call 2/5: Generating 5 variations... ✓ (10/25 total)
  API Call 3/5: Generating 5 variations... ✓ (15/25 total)
  API Call 4/5: Generating 5 variations... ✓ (20/25 total)
  API Call 5/5: Generating 5 variations... ✓ (25/25 total)

TOPIC 9/10: Psychological Manipulation
  Generating 25 VARIED samples...
  API calls needed: 5 (batch size: 5)
  API Call 1/5: Generating 5 variations... ✓ (5/25 total)
  API Call 2/5: Generating 5 variations... ✓ (10/25 total)
  API Call 3/5: Generating 5 variations... ✓ (15/25 total)
  API Call 4/5: Generating 5 variations... ✓ (20/25 total)
  API Call 5/5: Generating 5 variations... ✓ (25/25 total)

TOPIC 10/10: Childhood Vulnerability
  Generating 25 VARIED 

In [14]:
# Combine Class 3 datasets
class3_df = pd.concat([class3a_df, class3b_df], ignore_index=True)

print(f"\n✓ Class 3 (AI Mimicked) Complete Dataset:")
print(f"  - Total samples: {len(class3_df)}")
print(f"  - Austen-style samples: {len(class3a_df)}")
print(f"  - Dickens-style samples: {len(class3b_df)}")


✓ Class 3 (AI Mimicked) Complete Dataset:
  - Total samples: 500
  - Austen-style samples: 250
  - Dickens-style samples: 250


In [15]:
# Preview Class 3 samples
print("\n=== Sample AI Text (Mimicking Austen) ===")
print(class3a_df.iloc[239]['text'])

print("\n=== Sample AI Text (Mimicking Dickens) ===")
print(class3b_df.iloc[239]['text'])


=== Sample AI Text (Mimicking Austen) ===
To reflect upon the motivations driving the matrimonial market is to encounter a disheartening prevalence of calculation. Young men, no less than young women, are often guided by considerations of wealth and connections, rather than by any genuine affection. Mr. Darcy, though possessed of an undeniable integrity, initially viewed Elizabeth Bennet with a degree of condescension, owing to the inferior circumstances of her family. His subsequent alteration of opinion, while commendable, stemmed less from a sudden awakening of romantic sentiment, and more from a grudging acknowledgement of her wit and independent spirit – qualities which, he eventually conceded, might prove advantageous in a wife. The consequence, one fears, is a society where genuine connection is frequently overshadowed by pragmatic concerns.

=== Sample AI Text (Mimicking Dickens) ===
It is a melancholy truth, and one which weighs heavily upon the conscientious observer, that c

## Step 6: Combine All Classes and Create Final Dataset

In [16]:
# Standardize columns across all dataframes
def standardize_dataframe(df, source_class):
    """
    Ensures all dataframes have the same columns.
    """
    standard_df = pd.DataFrame()
    standard_df['text'] = df['text']
    standard_df['word_count'] = df['word_count']
    standard_df['class'] = source_class
    standard_df['author'] = df['author']

    # Add topic if it exists
    if 'topic' in df.columns:
        standard_df['topic'] = df['topic']
    else:
        standard_df['topic'] = 'original_literature'

    return standard_df

# Standardize all datasets
class1_standard = standardize_dataframe(class1_df, 'human')
class2_standard = standardize_dataframe(class2_df, 'ai_neutral')
class3_standard = standardize_dataframe(class3_df, 'ai_mimicked')

# Combine all three classes
final_dataset = pd.concat([
    class1_standard,
    class2_standard,
    class3_standard
], ignore_index=True)

# Shuffle the dataset
final_dataset = final_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

# Add a unique ID to each sample
final_dataset['sample_id'] = range(len(final_dataset))

print("="*60)
print("FINAL DATASET SUMMARY")
print("="*60)
print(f"\nTotal samples: {len(final_dataset)}")
print(f"\nClass distribution:")
print(final_dataset['class'].value_counts())
print(f"\nAuthor distribution:")
print(final_dataset['author'].value_counts())
print(f"\nWord count statistics:")
print(final_dataset.groupby('class')['word_count'].describe())

FINAL DATASET SUMMARY

Total samples: 1445

Class distribution:
class
ai_mimicked    500
ai_neutral     500
human          445
Name: count, dtype: int64

Author distribution:
author
AI                  500
Dickens_mimicked    250
Austen_mimicked     250
Dickens             227
Austen              218
Name: count, dtype: int64

Word count statistics:
             count        mean        std    min    25%    50%     75%    max
class                                                                        
ai_mimicked  500.0  132.186000  12.548719  100.0  123.0  131.0  141.00  168.0
ai_neutral   500.0  110.666000   9.219134   90.0  104.0  110.0  117.25  138.0
human        445.0  182.934831  16.581298  105.0  176.0  188.0  195.00  200.0


## Step 7: Save the Dataset

In [22]:

# Also save individual class datasets for reference
class1_standard.to_csv(os.path.join(PROJECT_PATH, 'class1_human.csv'), index=False)
class2_standard.to_csv(os.path.join(PROJECT_PATH,'class2_ai_neutral.csv'), index=False)
class3_standard.to_csv(os.path.join(PROJECT_PATH,'class3_ai_mimicked.csv'), index=False)
print("✓ Individual class datasets saved")

# Save topics list
with open(os.path.join(PROJECT_PATH,'extracted_topics.txt'), 'w') as f:
    for i, topic in enumerate(topics_list, 1):
        f.write(f"{i}. {topic}\n")
print("✓ Topics list saved to 'extracted_topics.txt'")

✓ Individual class datasets saved
✓ Topics list saved to 'extracted_topics.txt'


## Step 8: Basic Dataset Validation

In [23]:
# 1. Load the CSV Classes into DataFrames
class1_df = pd.read_csv(os.path.join(PROJECT_PATH, 'class1_human.csv'))
class2_df = pd.read_csv(os.path.join(PROJECT_PATH, 'class2_ai_neutral.csv'))
class3_df = pd.read_csv(os.path.join(PROJECT_PATH, 'class3_ai_mimicked.csv'))

# Combine all three classes
final_dataset = pd.concat([
    class1_df,
    class2_df,
    class3_df
], ignore_index=True)

# Shuffle the dataset
final_dataset = final_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

# Add a unique ID to each sample
final_dataset['sample_id'] = range(len(final_dataset))

final_dataset.to_csv(os.path.join(PROJECT_PATH,'final_dataset.csv'), index=False)

finalD = pd.read_csv(os.path.join(PROJECT_PATH,'final_dataset.csv'))
# 2. Load the Extracted Topics text file
with open(os.path.join(PROJECT_PATH, 'extracted_topics.txt'), 'r') as f:
    topics_list = [line.strip() for line in f.readlines() if line.strip()]

# Verification
print(f"Successfully loaded Class 1: {len(class1_df)} samples")
print(f"Successfully loaded Class 2: {len(class2_df)} samples")
print(f"Successfully loaded Class 3: {len(class3_df)} samples")
print(f"Successfully loaded {len(topics_list)} topics.")

Successfully loaded Class 1: 445 samples
Successfully loaded Class 2: 500 samples
Successfully loaded Class 3: 500 samples
Successfully loaded 20 topics.


In [21]:
# Check for any anomalies
print("\n" + "="*60)
print("DATASET VALIDATION")
print("="*60)

# Check for missing values
print("\nMissing values:")
print(final_dataset.isnull().sum())

# Check text length distribution
print("\nText length validation:")
too_short = final_dataset[final_dataset['word_count'] < 100]
too_long = final_dataset[final_dataset['word_count'] > 200]
print(f"  - Samples < 100 words: {len(too_short)}")
print(f"  - Samples > 200 words: {len(too_long)}")

# Check for duplicates
duplicates = final_dataset[final_dataset.duplicated(subset=['text'], keep=False)]
print(f"\nDuplicate texts: {len(duplicates)}")

print("\n✓ Task 0 Complete!")


DATASET VALIDATION

Missing values:
text          0
word_count    0
class         0
author        0
topic         0
sample_id     0
dtype: int64

Text length validation:
  - Samples < 100 words: 56
  - Samples > 200 words: 0

Duplicate texts: 7

✓ Task 0 Complete!


In [20]:
# Display a few random samples from each class for visual inspection
print("\n" + "="*60)
print("SAMPLE INSPECTION")
print("="*60)

for class_name in ['human', 'ai_neutral', 'ai_mimicked']:
    print(f"\n{'='*60}")
    print(f"CLASS: {class_name.upper()}")
    print(f"{'='*60}")
    samples = final_dataset[final_dataset['class'] == class_name].sample(n=2, random_state=42)

    for idx, row in samples.iterrows():
        print(f"\nAuthor: {row['author']} | Words: {row['word_count']}")
        print(f"Topic: {row['topic']}")
        print(f"Text: {row['text'][:400]}...")
        print("-" * 60)


SAMPLE INSPECTION

CLASS: HUMAN

Author: Austen | Words: 158
Topic: original_literature
Text: success, and what we are to talk of next I cannot imagine.” "What think you of books?” said he, smiling. "Books--oh no!--I am sure we never read the same, or not with the same feelings.” "I am sorry you think so; but if that be the case, there can at least be no want of subject. We may compare our different opinions.” "No--I cannot talk of books in a ball-room; my head is always full of something ...
------------------------------------------------------------

Author: Austen | Words: 199
Topic: original_literature
Text: many years; but I very well remember that I never liked her, and that her manners were dictatorial and insolent. She has the reputation of being remarkably sensible and clever; but I rather believe she derives part of her abilities from her rank and fortune, part from her authoritative manner, and the rest from the pride of her nephew, who chooses that everyone connected with